# Notebook 01 — Exploración interactiva del conteo ONPE EG2026

Este notebook está diseñado para ser usado **en vivo durante una audiencia** si alguien pregunta "¿cómo llegó a ese número?". Cada celda es reproducible desde las capturas en `captures/`.

**Flujo:**
1. Cargar la última captura y verificar hashes.
2. Explorar los datos regionales.
3. Re-ejecutar cada test del informe y mostrar cómo se calcula.
4. Generar visualizaciones adicionales a demanda.

In [ ]:
# Setup
import json, hashlib, sys
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

print(f'ROOT = {ROOT}')
captures = sorted([p for p in (ROOT/'captures').iterdir() if p.is_dir() and p.name[0].isdigit()])
print(f'Capturas disponibles: {len(captures)}')
for c in captures: print(f'  {c.name}')

## 1. Verificación de integridad de la última captura

Re-calculamos hashes y comparamos con el manifiesto.  
Si algo cambió después de la captura, el test falla.

In [ ]:
capture = captures[-1]
manifest = json.loads('[' + ','.join((capture/'MANIFEST.jsonl').read_text().strip().split('\n')) + ']')

def sha256(p):
    h = hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''): h.update(chunk)
    return h.hexdigest()

seen = set()
for e in manifest:
    if e['endpoint'] in seen: continue
    seen.add(e['endpoint'])
    actual = sha256(capture/e['local_path'])
    mark = '✓' if actual == e['sha256'] else '✗ MISMATCH'
    print(f"  {mark}  {e['endpoint']:<15}  {e['bytes']:>7}B  {actual[:16]}...")

## 2. Carga del dataset consolidado

In [ ]:
df = pd.read_csv(ROOT/'data/processed/regiones.csv')
meta = json.loads((ROOT/'data/processed/meta.json').read_text())

print(f"Corte: {meta['pct_global']}%  ·  {meta['capture_ts_utc']}")
print(f"Margen Sánchez − RLA: {meta['margen_sanch_rla_votos']:+,}  ({meta['margen_sanch_rla_pct']:+.3f} pts)\n")
df.head(10)

## 3. ¿Dónde están las actas impugnadas?

Exploración interactiva — cambia el threshold para ver qué regiones quedan fuera.

In [ ]:
threshold = 10.0  # cambia este valor
df[df['tasa_impugnacion'] >= threshold][['name','totalActas','enviadasJee','tasa_impugnacion']].sort_values('tasa_impugnacion', ascending=False)

## 4. Simulación interactiva: ¿cuántos votos necesita RLA en el Extranjero?

In [ ]:
# Extranjero tiene 613 actas impugnadas × ~181 votos/acta = ~111k votos en disputa
# Si todas esas actas se anularan, ¿qué pasa con el margen?

margen = meta['margen_sanch_rla_votos']
extranjero = df[df['name'].str.lower().str.contains('extr', na=False)]
if len(extranjero):
    e = extranjero.iloc[0]
    votos_por_acta = 181.9  # promedio nacional
    votos_extranjero_jee = e['enviadasJee'] * votos_por_acta
    # Si esos votos se distribuyeran como la votación actual en el extranjero:
    votos_rla_ext = votos_extranjero_jee * e['rla'] / 100
    votos_sanch_ext = votos_extranjero_jee * e['sanch'] / 100
    delta = votos_rla_ext - votos_sanch_ext
    print(f"Actas JEE Extranjero: {int(e['enviadasJee'])}")
    print(f"Votos en juego ahí: ~{int(votos_extranjero_jee):,}")
    print(f"Bajo distribución similar a extranjero procesado:")
    print(f"  RLA gana ~{int(votos_rla_ext):,}")
    print(f"  Sánchez gana ~{int(votos_sanch_ext):,}")
    print(f"  Delta a favor de RLA: {int(delta):+,}")
    print(f"\nMargen actual Sánchez-RLA: {margen:+,}")
    print(f"Margen si se regulariza solo Extranjero: {margen - int(delta):+,}")

## 5. Benford en vivo

Aplicamos Benford a los votos por candidato × región y mostramos el histograma.

In [ ]:
def first_digit(v):
    s = str(int(v)).lstrip('0')
    return int(s[0]) if s else None

pool = []
for c in ['fuji_v','rla_v','nieto_v','belm_v','sanch_v']:
    pool.extend(df[c].tolist())
digits = [first_digit(v) for v in pool if v > 0]
digits = [d for d in digits if d is not None]

counts = {d: digits.count(d) for d in range(1,10)}
expected = {d: len(digits) * np.log10(1 + 1/d) for d in range(1,10)}

fig, ax = plt.subplots(figsize=(9,4))
dg = list(range(1,10))
ax.bar([d-0.2 for d in dg], [counts[d] for d in dg], 0.4, label='Observado', color='#2980b9')
ax.bar([d+0.2 for d in dg], [expected[d] for d in dg], 0.4, label='Benford esperado', color='#e67e22')
ax.set_xticks(dg); ax.set_xlabel('Primer dígito'); ax.set_ylabel('Frecuencia')
ax.legend(); ax.set_title('Benford — 5 candidatos × 26 regiones'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

chi2 = sum((counts[d]-expected[d])**2/expected[d] for d in dg)
p = 1 - stats.chi2.cdf(chi2, df=8)
print(f'χ² = {chi2:.2f}, p = {p:.3f} → {"conforme" if p >= 0.05 else "desvía"} a Benford')